# **Import**

In [ ]:
!pip install scikit-bio
!pip install scikit-posthocs

In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import scikit_posthocs as sp
from scipy.stats import ttest_ind_from_stats, wilcoxon,shapiro

# **Testes**

In [ ]:
data1= pd.read_csv("/content/dados.csv") # CSV do tempo de inferência

Shapiro

In [ ]:
caminho_csv = 'TempolotesTemp1.csv'

df = pd.read_csv(caminho_csv)

resultados = []


for coluna in df.select_dtypes(include='number').columns:
    dados = df[coluna].dropna()

    if len(dados) < 3:
        resultados.append({
            'Coluna': coluna,
            'Estatística': None,
            'Valor-p': None,
            'Normalidade': 'Dados insuficientes'
        })
        continue

    stat, p = shapiro(dados)

    normalidade = 'Normal (p > 0.05)' if p > 0.05 else ' Não normal (p ≤ 0.05)'

    resultados.append({
        'Coluna': coluna,
        'Estatística': round(stat, 8),
        'Valor-p': round(p, 8),
        'Normalidade': normalidade
    })

tabela_resultados = pd.DataFrame(resultados)

Verificando normalidade

In [ ]:
def distribuicao(caminho_csv):

    pasta_graficos = 'graficos_distribuicaoLotes1'
    os.makedirs(pasta_graficos, exist_ok=True)


    df = pd.read_csv(caminho_csv)


    colunas_numericas = df.select_dtypes(include='number').columns

    for coluna in colunas_numericas:
        dados = df[coluna].dropna()

        if len(dados) < 3:
            print(f' Coluna {coluna} tem menos de 3 valores. Pulando...')
            continue


        plt.figure(figsize=(12, 5))


        plt.subplot(1, 2, 1)
        sns.histplot(dados, kde=True, bins=30, color='skyblue')
        plt.title(f'Distribuição: {coluna}')
        plt.xlabel(coluna)
        plt.ylabel('Frequência')


        plt.subplot(1, 2, 2)
        stats.probplot(dados, dist="norm", plot=plt)
        plt.title(f'Q-Q Plot: {coluna}')

        plt.tight_layout()


        nome_arquivo = os.path.join(pasta_graficos, f'{coluna}_distribuicao.png')
        plt.savefig(nome_arquivo)
        plt.close()

        print(f'Gráfico salvo: {nome_arquivo}')


# RQ1

In [ ]:
posthoc_folder = "posthoc_kruskal1"
os.makedirs(posthoc_folder, exist_ok=True)

data_long = pd.melt(
    data1.reset_index(),
    id_vars=['index'],
    value_vars=data1.columns
)
data_long.columns = ['Observation', 'Group', 'Value']

dunn_result = sp.posthoc_dunn(
    data_long,
    val_col='Value',
    group_col='Group',
    p_adjust='fdr_bh'
)

plt.figure(figsize=(7.7, 6.0))

# Gerando Heatmap
ax = sns.heatmap(
    dunn_result,
    annot=True,
    fmt=".1e",
    cmap="coolwarm",
    linewidths=2,
    linecolor='black',
    square=False,
    annot_kws={
        "size": 12,
        "weight": "bold"
    },
    cbar=False
)


ax.set_aspect('auto')

mesh = ax.collections[0]
facecolors = mesh.get_facecolors()
for text, facecolor in zip(ax.texts, facecolors):
    r, g, b, _ = facecolor
    luminance = 0.299*r + 0.587*g + 0.114*b
    text.set_color('white' if luminance < 0.5 else 'black')

ax.tick_params(axis='both', labelsize=14, width=3)
plt.xticks(rotation=45, ha='right', fontweight='bold')
plt.yticks(fontweight='bold')


plt.tight_layout(pad=0)

heatmap_path = os.path.join(posthoc_folder, "0LotesDunn_.png")
plt.savefig(
    heatmap_path,
    dpi=300,
    bbox_inches='tight',
    pad_inches=0
)

plt.show()

# RQ2

Comparativo dos pares dos modelos tanto para as instâncias quanto para os lotes. MLP0 VS MLP1, RANDOM FOREST0 VS  RANDOM FOREST01....

In [ ]:
mean_benign = pd.read_csv("Media_Lotescolunas0.csv", index_col=0)
mean_malicious = pd.read_csv("Média_Lotescolunas1.csv", index_col=0)
std_benign = pd.read_csv("Desvio_Lotescolunas0.csv", index_col=0)
std_malicious = pd.read_csv("Desvio_Lotescolunas1.csv", index_col=0)

n = 500

algorithms = set(mean_benign.index) & set(mean_malicious.index) & set(std_benign.index) & set(std_malicious.index)

results = []

for alg in algorithms:

    m_benign = float(mean_benign.loc[alg].values[0])
    m_malicious = float(mean_malicious.loc[alg].values[0])
    s_benign = float(std_benign.loc[alg].values[0])
    s_malicious = float(std_malicious.loc[alg].values[0])

    t_stat, p_val = ttest_ind_from_stats(
        mean1=m_benign, std1=s_benign, nobs1=n,
        mean2=m_malicious, std2=s_malicious, nobs2=n,
        equal_var=False
    )

    significance = "Significant" if p_val < 0.05 else "Not Significant"

    p_val_formatted = f"{p_val:.2e}"

    results.append({
        "Algorithm": alg,
        "t-statistic": round(t_stat, 4),
        "p-value": p_val_formatted,
        "Conclusion": significance
    })


results_df = pd.DataFrame(results)


results_df["p-value_float"] = results_df["p-value"].astype(float)
results_df = results_df.sort_values(by="p-value_float")
results_df = results_df.drop(columns="p-value_float")


results_df.to_csv("inference_comparison_results0_1.csv", index=False)

print("Results saved to 'inference_comparison_results0.csv'")


Comparação dos pares de modelos Instância VS LOTE. Ex: MLP0_instância VS MLP0_Lote

In [ ]:
import pandas as pd

mean_benign = pd.read_csv("medias_colunas0.csv", index_col=0) Instância
mean_malicious = pd.read_csv("Media_Lotescolunas0.csv", index_col=0) Lote
std_benign = pd.read_csv("Devios_colunas00.csv", index_col=0)  Instância
std_malicious = pd.read_csv("Desvio_Lotescolunas0.csv", index_col=0) Lote

n = 500

algorithms = set(mean_benign.index) & set(mean_malicious.index) & set(std_benign.index) & set(std_malicious.index)

results = []

for alg in algorithms:
    # Extrai valores
    m_benign = float(mean_benign.loc[alg].values[0])
    m_malicious = float(mean_malicious.loc[alg].values[0])
    s_benign = float(std_benign.loc[alg].values[0])
    s_malicious = float(std_malicious.loc[alg].values[0])


    t_stat, p_val = ttest_ind_from_stats(
        mean1=m_benign, std1=s_benign, nobs1=n,
        mean2=m_malicious, std2=s_malicious, nobs2=n,
        equal_var=False
    )

    significance = "Significant" if p_val < 0.05 else "Not Significant"


    p_val_formatted = f"{p_val:.2e}"

    results.append({
        "Algorithm": alg,
        "t-statistic": round(t_stat, 4),
        "p-value": p_val_formatted,
        "Conclusion": significance
    })

results_df = pd.DataFrame(results)

results_df["p-value_float"] = results_df["p-value"].astype(float)
results_df = results_df.sort_values(by="p-value_float")
results_df = results_df.drop(columns="p-value_float")  # Remove coluna auxiliar

results_df.to_csv("inference_comparison_results00.csv", index=False)

print("Results saved to 'inference_comparison_results0000.csv'")

Instâncias

Aqui apenas as Médias são comparadas

In [ ]:

# Data
mean0 = [824.124,817.068,933.99,853.978,4800.5,805.7,799.66,823.858,7159.08]
mean1 = [809.326,813.092,933.148,841.122,4740.0,793.818,797.68,812.294,7154.12]


df = pd.DataFrame({
    'Group': ['Class 0'] * len(mean0) + ['Class 1'] * len(mean1),
    'Value': mean0 + mean1
})

df.to_csv('population_comparison.csv', index=False)
w_stat, p_val = wilcoxon(mean1, mean0)


print(f"Wilcoxon Statistic (W): {w_stat:.10f}")
print(f"P-value: {p_val:.10f}")


summary = pd.DataFrame({
    'Statistic': ['W', 'p'],
    'Value': [round(w_stat, 10), round(p_val, 10)]
})

fig, ax = plt.subplots(figsize=(6, 5))
ax.axis('tight')
ax.axis('off')
table = ax.table(cellText=df.values,
                 colLabels=df.columns,
                 loc='center',
                 cellLoc='center')

plt.title("Comparison of Class 0 and 1", fontsize=14, weight='bold', pad=20)
plt.savefig('population_data_table.png', dpi=300, bbox_inches='tight')
plt.close()


fig2, ax2 = plt.subplots(figsize=(4.5, 2))
ax2.axis('tight')
ax2.axis('off')
table2 = ax2.table(cellText=summary.values,
                   colLabels=summary.columns,
                   loc='center',
                   cellLoc='center')

plt.title("Wilcoxon test result comparing instances of classes 0 and 1", fontsize=10, weight='bold', pad=10)
plt.savefig('wilcoxon_test_result.png', dpi=300, bbox_inches='tight')
plt.show()


Lotes

In [ ]:
mean0 = [1440.0, 867.4, 1118.0, 1010.0, 6696.0, 844.6, 842.8, 858.8, 9544.0]
mean1 = [1440.0, 855.0, 1122.0, 1014.0, 6756.0, 852.4, 846.0, 863.2, 9504.0]


df = pd.DataFrame({
    'Group': ['Class 0'] * len(mean0) + ['Class 1'] * len(mean1),
    'Value': mean0 + mean1
})


df.to_csv('population_comparison.csv', index=False)


w_stat, p_val = wilcoxon(mean1, mean0)


summary = pd.DataFrame({
    'Statistic': ['W', 'p'],
    'Value': [round(w_stat, 4), round(p_val, 4)]
})


fig, ax = plt.subplots(figsize=(6, 5))
ax.axis('tight')
ax.axis('off')
table = ax.table(cellText=df.values,
                 colLabels=df.columns,
                 loc='center',
                 cellLoc='center')

plt.title("Comparison of Class 0 and 1", fontsize=14, weight='bold', pad=20)
plt.savefig('population_data_table.png', dpi=300, bbox_inches='tight')
plt.close()


fig2, ax2 = plt.subplots(figsize=(4.5, 2))
ax2.axis('tight')
ax2.axis('off')
table2 = ax2.table(cellText=summary.values,
                   colLabels=summary.columns,
                   loc='center',
                   cellLoc='center')

plt.title("Wilcoxon test result comparing the batches of classes 0 and 1", fontsize=10, weight='bold', pad=10)
plt.savefig('wilcoxon_test_result.png', dpi=300, bbox_inches='tight')
plt.show()


# RQ3


Classes misturadas 01

In [ ]:
mean01instan = [820.942,821.384,944.248,850.938,4832.32,805.11,806.046,821.008,7163.88]
mean01batch = [850.8,818.6,974.2,867.4,5324.0,787.6,801.2,811.2,9424.0]

df = pd.DataFrame({
    'Group': ['Class 01 - Instances'] * len(mean01instan) + ['Class 01 - Batches'] * len(mean01batch),
    'Value': mean01instan + mean01batch
})


df.to_csv('population_comparisonbatch_instan01.csv', index=False)


w_stat, p_val = wilcoxon(mean01instan, mean01batch)

print(f"Wilcoxon Statistic (W): {w_stat:.10f}")
print(f"P-value: {p_val:.10f}")

summary = pd.DataFrame({
    'Statistic': ['W', 'p'],
    'Value': [round(w_stat, 10), round(p_val, 10)]
})


fig, ax = plt.subplots(figsize=(6, 5))
ax.axis('tight')
ax.axis('off')
table = ax.table(cellText=df.values,
                 colLabels=df.columns,
                 loc='center',
                 cellLoc='center')

plt.title("Comparison of instances and batches of classes 0 and 1", fontsize=14, weight='bold', pad=20)
plt.savefig('population_data_table.png', dpi=300, bbox_inches='tight')
plt.close()


fig2, ax2 = plt.subplots(figsize=(4.5, 2))
ax2.axis('tight')
ax2.axis('off')
table2 = ax2.table(cellText=summary.values,
                   colLabels=summary.columns,
                   loc='center',
                   cellLoc='center')

plt.title("Result of the Wilcoxon test comparing instances and batches of classes 0 and 1", fontsize=10, weight='bold', pad=10)
plt.savefig('wilcoxon_test_resultbatch_instanc.png', dpi=300, bbox_inches='tight')
plt.show()


Classe 0

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon

mean0instan = [824.124,817.068,933.99,853.978,4800.5,805.7,799.66,823.858,7159.08]
mean0batch = [1440.0, 867.4, 1118.0, 1010.0, 6696.0, 844.6, 842.8, 858.8, 9544.0]

df = pd.DataFrame({
    'Group': ['Class 0'] * len(mean0instan) + ['Class 1'] * len(mean0batch),
    'Value': mean0instan + mean0batch
})


df.to_csv('population_comparisonbatch_instan.csv', index=False)


w_stat, p_val = wilcoxon(mean0instan, mean0batch)

print(f"Wilcoxon Statistic (W): {w_stat:.10f}")
print(f"P-value: {p_val:.10f}")

summary = pd.DataFrame({
    'Statistic': ['W', 'p'],
    'Value': [round(w_stat, 10), round(p_val, 10)]
})


fig, ax = plt.subplots(figsize=(6, 5))
ax.axis('tight')
ax.axis('off')
table = ax.table(cellText=df.values,
                 colLabels=df.columns,
                 loc='center',
                 cellLoc='center')

plt.title("Comparison of instances and batches for class 0", fontsize=14, weight='bold', pad=20)
plt.savefig('population_data_table.png', dpi=300, bbox_inches='tight')
plt.close()


fig2, ax2 = plt.subplots(figsize=(4.5, 2))
ax2.axis('tight')
ax2.axis('off')
table2 = ax2.table(cellText=summary.values,
                   colLabels=summary.columns,
                   loc='center',
                   cellLoc='center')

plt.title("Result of the Wilcoxon test comparing instances and batches of class 0", fontsize=10, weight='bold', pad=10)
plt.savefig('wilcoxon_test_resultbatch_instanc.png', dpi=300, bbox_inches='tight')
plt.show()

Classe 1

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon

mean1instan = [809.326,813.092,933.148,841.122,4740.0,793.818,797.68,812.294,7154.12]
mean1batch =  [1440.0, 855.0, 1122.0, 1014.0, 6756.0, 852.4, 846.0, 863.2, 9504.0]

df = pd.DataFrame({
    'Group': ['Class 0'] * len(mean1instan) + ['Class 1'] * len(mean1batch),
    'Value': mean1instan + mean1batch
})


df.to_csv('population_comparisonbatch_instan.csv', index=False)


w_stat, p_val = wilcoxon(mean1instan, mean1batch)

print(f"Wilcoxon Statistic (W): {w_stat:.10f}")
print(f"P-value: {p_val:.10f}")

summary = pd.DataFrame({
    'Statistic': ['W', 'p'],
    'Value': [round(w_stat, 10), round(p_val, 10)]
})


fig, ax = plt.subplots(figsize=(6, 5))
ax.axis('tight')
ax.axis('off')
table = ax.table(cellText=df.values,
                 colLabels=df.columns,
                 loc='center',
                 cellLoc='center')

plt.title("Comparison of instances and batches for class 1", fontsize=14, weight='bold', pad=20)
plt.savefig('population_data_table.png', dpi=300, bbox_inches='tight')
plt.close()


fig2, ax2 = plt.subplots(figsize=(4.5, 2))
ax2.axis('tight')
ax2.axis('off')
table2 = ax2.table(cellText=summary.values,
                   colLabels=summary.columns,
                   loc='center',
                   cellLoc='center')

plt.title("Result of the Wilcoxon test comparing instances and batches of class 1", fontsize=10, weight='bold', pad=10)
plt.savefig('wilcoxon_test_resultbatch_instanc1.png', dpi=300, bbox_inches='tight')
plt.show()
